In [1]:
import math
import ast
import ipynbname
import pandas as pd
import numpy as np
import optuna
from Functions.AutoCloud_V2 import *
from Functions.DataCloud_V2 import *
from Functions.Utils import *
from Functions.Graphs import *
from Functions.TedaGraphs import *
from Functions.Utils_OPT import *
from Functions.TedaOptimize_QN import *

FileName = ipynbname.name()
out_path = f'Optimization\\eDRTLO_QN\\multi\\Optimization.csv'
RS = pd.read_excel(r'Dataset\RS.xlsx')
HI = pd.read_excel(r'Dataset\HI.xlsx')
sig = HI['PC1'].values

In [2]:
dfs = Optimize(FileName=FileName[:-4],OptDim=1,OptSampler=None,OptPrune=False,
                         n_study=1,timeout=60,n_trials=1e1,patience=None,
                         mS=[2.0,4.5],nRS=[1,70],mdS=[1,1],actS=[0,1])

iteration: 1


[I 2026-08-14 08:40:16,520] A new study created in memory with name: no-name-bfde733a-26b2-4eab-9e5f-9f087afa81c4
[I 2026-08-14 08:40:16,522] Trial 0 pruned. 
[I 2026-08-14 08:40:16,524] Trial 1 pruned. 
[I 2026-08-14 08:40:16,525] Trial 2 pruned. 
[I 2026-08-14 08:40:16,527] Trial 3 pruned. 
[I 2026-08-14 08:40:16,529] Trial 4 pruned. 
[I 2026-08-14 08:40:16,772] Trial 5 finished with value: 2.311132397801352 and parameters: {'m': 4.5, 'nI': 16, 'n_layers': 3, 'nR_layer_0': 9, 'nR_layer_1': 38, 'nR_layer_2': 49, 'nO': 16, 'N': 4, 'τ': 6, 'mode': 1, 'act': 0, 'mO': 12}. Best is trial 5 with value: 2.311132397801352.
[I 2026-08-14 08:40:16,774] Trial 6 pruned. 
[I 2026-08-14 08:40:16,775] Trial 7 pruned. 
[I 2026-08-14 08:40:16,778] Trial 8 pruned. 
[I 2026-08-14 08:40:16,779] Trial 9 pruned. 


In [ ]:
for i in range(1,6):
    dfs = Optimize(FileName=FileName[:-4],OptDim=2,OptSampler=None,OptPrune=False,
                            n_study=6,timeout=1680,n_trials=2e4,patience=2e3,
                            mS=[2.0,4.5],nLS=[i,i],nRS=[1,70],mdS=[1,1],actS=[0,1])

In [3]:
df = dfs[0]
#df = df[(df.iloc[:,0] <= 0.07)]
params_list = df.values[:,-9:]
df

,MAPE_RUL*MAPE_HI,m,nI,nR,nO,mO,N,TAU,past/ahead,activation
0,1.522142,4.00,7,"[32, 20, 55]",4,5,3,17.591303,1,0
1,1.482010,3.00,20,[57],16,12,9,6.553969,1,0
2,1.435604,3.25,7,[12],3,1,5,15.461657,1,1
3,1.279721,2.25,8,"[57, 36]",14,5,1,24.538093,1,0
4,0.478720,2.00,11,"[52, 41]",13,4,9,9.728509,1,0
...,...,...,...,...,...,...,...,...,...,...
91,0.334883,3.50,10,"[23, 22, 3, 44, 65]",7,1,1,25.000000,1,0
92,0.332072,4.25,10,"[26, 40, 16, 56, 52]",2,9,1,14.000000,1,0
93,0.267663,2.75,3,"[26, 43, 58, 2, 42]",19,0,1,5.000000,1,0
94,0.177106,4.00,2,"[46, 11, 53, 31, 67]",5,2,1,21.000000,1,0


In [ ]:
tedas = []
for i,params in enumerate(params_list):
    m,nI,nR,nO,mO,N1,τ,mode,act = params
    X, Y, Z = PrepareData(RS, HI, nI, nO, mO, 1, mode)
    if isinstance(nR, str): nR = ast.literal_eval(nR)

    teda=AutoCloud(m=m,nI=len(Y[0]),nR=nR,nO=nO+mO,ηS=[N1],mode=mode,act=act,
                tau=τ,rho=0.03,eol=0.3,ref=len(sig)-nI+1,wtaG=True,wtaP=True) 
    for j,_ in enumerate(X[:]):
        teda.run(X[j])
        teda.RUL_Prediction(Y[j],mode='single',lim=len(sig)-nI+1,show=False)
        teda.Adapt(Y[j],Z[j])

    teda.c = np.append(teda.c,teda.gm)  
    tedas.append(teda)
    #PlotSeriesPLY(ySeries=[teda.wape_HI_hist,teda.wape_RUL_hist])
    

In [ ]:
PlotDSI_3D_PLT(tedas[0])